# PySpark API: RDD and DataFrame Operations


This notebook demonstrates core PySpark operations using RDD and DataFrame APIs, including transformations, actions, filtering, aggregation, sorting, and data processing with the Titanic dataset.

In [ ]:
!pip install -q pyspark

import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SPARK_API") \
    .getOrCreate()

sc = spark.sparkContext


In [ ]:
titanic_df = spark.read.json("data/titanic.json")

titanic_df = titanic_df.select(
    "PassengerId",
    "Survived",
    "PassengerClass",
    "PassengerName",
    "Sex",
    "Age",
    "SiblingsAndSpouses",
    "ParentsAndChildren",
    "Ticket",
    "Fare",
    "Cabin",
    "Embarked"
)

titanic_df.write.mode("overwrite").parquet("titanic_parquet/")

## RDD API

In [ ]:
titanic_rdd = spark.read.format("parquet").load("titanic_parquet").rdd

In [ ]:
# Compare survival rates across passenger classes

first_class_survival_rate = (
    titanic_rdd.filter(lambda row: row[2] == "1" and row[1] == "1").count()
    / titanic_rdd.filter(lambda row: row[2] == "1").count()
)

second_class_survival_rate = (
    titanic_rdd.filter(lambda row: row[2] == "2" and row[1] == "1").count()
    / titanic_rdd.filter(lambda row: row[2] == "2").count()
)

third_class_survival_rate = (
    titanic_rdd.filter(lambda row: row[2] == "3" and row[1] == "1").count()
    / titanic_rdd.filter(lambda row: row[2] == "3").count()
)

print(f"First class survival rate: {first_class_survival_rate:.2%}")
print(f"Second class survival rate: {second_class_survival_rate:.2%}")
print(f"Third class survival rate: {third_class_survival_rate:.2%}")

First class survival rate: 62.96%
Second class survival rate: 47.28%
Third class survival rate: 24.24%


In [ ]:
# What was the ratio of women to men in first class? (.count() function)

first_class_male_count = titanic_rdd \
                         .filter(lambda row: row[2] == "1" and row[4] == "male") \
                         .count()
first_class_female_count = titanic_rdd \
                           .filter(lambda row: row[2] == "1" and row[4] == "female") \
                           .count()
total_first_class_count = titanic_rdd \
                           .filter(lambda row: row[2] == "1") \
                           .count()
print(f"male: {first_class_male_count/total_first_class_count}")
print(f"female: {first_class_female_count/total_first_class_count}")

male: 0.5648148148148148
female: 0.4351851851851852


In [ ]:
# Extract passenger names using RDD map()
names = titanic_rdd.map(lambda x: x[3])
print(names.take(10))

['Braund, Mr. Owen Harris', 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)', 'Heikkinen, Miss. Laina', 'Futrelle, Mrs. Jacques Heath (Lily May Peel)', 'Allen, Mr. William Henry', 'Moran, Mr. James', 'McCarthy, Mr. Timothy J', 'Palsson, Master. Gosta Leonard', 'Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)', 'Nasser, Mrs. Nicholas (Adele Achem)']


In [ ]:
# Count the total number of passengers
passenger_count = titanic_rdd.count()
print(passenger_count)

891


In [ ]:
# Count first-class passengers
titanic_first_class_count = titanic_rdd.filter(lambda row: row[2] == "1").count()
print(titanic_first_class_count)

216


In [ ]:
# Count passengers who did not survive
titanic_rdd.map(lambda row: int(row[1])) \
           .filter(lambda survived: survived == 0) \
           .count()

549

### Map and Reduce Operations

Using `map()` for transformation and `reduce()` for aggregation on PySpark RDDs.

In [ ]:
# Find the youngest passenger age using map() and reduce()

youngest_age = titanic_rdd \
    .map(lambda row: row[5]) \
    .filter(lambda age: age not in (None, "")) \
    .map(lambda age: float(age)) \
    .reduce(lambda age1, age2: min(age1, age2))

print(f"Youngest passenger age: {youngest_age}")

Youngest passenger age: 0.42


In [ ]:
# Count the total number of passengers using map() and reduce()

traveller_count = titanic_rdd \
    .map(lambda row: 1) \
    .reduce(lambda count1, count2: count1 + count2)

print(f"Traveller count: {traveller_count}")

Traveller count: 891


In [ ]:
# Count passengers with unknown age using map() and reduce()

unknown_age_count = titanic_rdd \
    .map(lambda row: row[5]) \
    .filter(lambda age: age in (None, "")) \
    .map(lambda age: 1) \
    .reduce(lambda count1, count2: count1 + count2)

print(f"Unknown age count: {unknown_age_count}")

Unknown age count: 177


In [ ]:
# Convert passenger names to uppercase using map()

passenger_names = titanic_rdd \
    .map(lambda row: row[3]) \
    .map(lambda name: name.upper()) \
    .take(10)

print(passenger_names)

['BRAUND, MR. OWEN HARRIS', 'CUMINGS, MRS. JOHN BRADLEY (FLORENCE BRIGGS THAYER)', 'HEIKKINEN, MISS. LAINA', 'FUTRELLE, MRS. JACQUES HEATH (LILY MAY PEEL)', 'ALLEN, MR. WILLIAM HENRY', 'MORAN, MR. JAMES', 'MCCARTHY, MR. TIMOTHY J', 'PALSSON, MASTER. GOSTA LEONARD', 'JOHNSON, MRS. OSCAR W (ELISABETH VILHELMINA BERG)', 'NASSER, MRS. NICHOLAS (ADELE ACHEM)']


In [ ]:
# Find the oldest passenger age using map() and reduce()

oldest_age = titanic_rdd \
    .map(lambda row: row[5]) \
    .filter(lambda age: age not in (None, "")) \
    .map(lambda age: float(age)) \
    .reduce(lambda age1, age2: max(age1, age2))

print(f"Oldest passenger age: {oldest_age}")

Oldest passenger age: 80.0


In [ ]:
# Convert passenger names to lowercase using map()

passenger_names_lower = titanic_rdd \
    .map(lambda row: row[3]) \
    .map(lambda name: name.lower()) \
    .take(10)

print(passenger_names_lower)

['braund, mr. owen harris', 'cumings, mrs. john bradley (florence briggs thayer)', 'heikkinen, miss. laina', 'futrelle, mrs. jacques heath (lily may peel)', 'allen, mr. william henry', 'moran, mr. james', 'mccarthy, mr. timothy j', 'palsson, master. gosta leonard', 'johnson, mrs. oscar w (elisabeth vilhelmina berg)', 'nasser, mrs. nicholas (adele achem)']


## Sorting and Selecting RDD Data

Using `sortBy()` and `take()` to sort RDD data and retrieve a limited number of results.

In [ ]:
# Retrieve the 10 youngest known passenger ages

first_10_ages = titanic_rdd \
    .map(lambda row: row[5]) \
    .filter(lambda age: age not in (None, "")) \
    .map(lambda age: float(age)) \
    .sortBy(lambda age: age) \
    .take(10)

print(first_10_ages)

[0.42, 0.67, 0.75, 0.75, 0.83, 0.83, 0.92, 1.0, 1.0, 1.0]


In [ ]:
# Retrieve the 5 oldest known passenger ages

first_5_ages = titanic_rdd \
    .map(lambda row: row[5]) \
    .filter(lambda age: age not in (None, "")) \
    .map(lambda age: float(age)) \
    .sortBy(lambda age: age, ascending=False) \
    .take(5)

print(first_5_ages)

[80.0, 74.0, 71.0, 71.0, 70.5]


## RDD Partitioning

Using RDD partitioning to control how data is distributed for parallel processing in PySpark.

In [ ]:
# Check the current number of partitions

current_partitions = titanic_rdd.getNumPartitions()
print(f"Current partitions: {current_partitions}")

Current partitions: 1


In [ ]:
# Repartition the RDD into 15 partitions

repartitioned_rdd = titanic_rdd.repartition(15)

print(f"Partitions after repartitioning: {repartitioned_rdd.getNumPartitions()}")

Partitions after repartitioning: 15


In [ ]:
# Check the number of records in each partition

partition_sizes = titanic_rdd \
    .repartition(3) \
    .mapPartitions(lambda partition: [len(list(partition))]) \
    .collect()

print(f"Records per partition: {partition_sizes}")

Records per partition: [300, 300, 291]


## Distinct Values in RDDs

Using `distinct()` to identify unique values within PySpark RDD data.

In [ ]:
# Display the unique passenger classes

passenger_classes = titanic_rdd \
    .map(lambda row: row[2]) \
    .distinct() \
    .sortBy(lambda passenger_class: passenger_class) \
    .collect()

print(passenger_classes)

['1', '2', '3']


## Combining RDDs

Using `union()` to combine multiple PySpark RDDs into a single RDD.

In [ ]:
# Combine male and female passenger RDDs using union()

male_rdd = titanic_rdd.filter(lambda row: row[4] == "male")
female_rdd = titanic_rdd.filter(lambda row: row[4] == "female")

male_and_female = male_rdd.union(female_rdd)

print(f"Combined passenger count: {male_and_female.count()}")
print(male_and_female.take(5))

Combined passenger count: 891
[Row(PassengerId='1', Survived='0', PassengerClass='3', PassengerName='Braund, Mr. Owen Harris', Sex='male', Age='22', SiblingsAndSpouses='1', ParentsAndChildren='0', Ticket='A/5 21171', Fare='7.25', Cabin='', Embarked='S'), Row(PassengerId='5', Survived='0', PassengerClass='3', PassengerName='Allen, Mr. William Henry', Sex='male', Age='35', SiblingsAndSpouses='0', ParentsAndChildren='0', Ticket='373450', Fare='8.05', Cabin='', Embarked='S'), Row(PassengerId='6', Survived='0', PassengerClass='3', PassengerName='Moran, Mr. James', Sex='male', Age='', SiblingsAndSpouses='0', ParentsAndChildren='0', Ticket='330877', Fare='8.4583', Cabin='', Embarked='Q'), Row(PassengerId='7', Survived='0', PassengerClass='1', PassengerName='McCarthy, Mr. Timothy J', Sex='male', Age='54', SiblingsAndSpouses='0', ParentsAndChildren='0', Ticket='17463', Fare='51.8625', Cabin='E46', Embarked='S'), Row(PassengerId='8', Survived='0', PassengerClass='3', PassengerName='Palsson, Mast

## Key-Based Aggregation

Using `reduceByKey()` and `groupByKey()` to aggregate values by key in PySpark RDDs.

In [ ]:
# Count passengers in each class using reduceByKey()

class_travelers = titanic_rdd \
    .filter(lambda row: row[2] not in (None, "")) \
    .map(lambda row: (row[2], 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortByKey() \
    .collect()

print(class_travelers)

[('1', 216), ('2', 184), ('3', 491)]


In [ ]:
# Count passengers in each class using groupByKey()

class_travelers_grouped = titanic_rdd \
    .filter(lambda row: row[2] not in (None, "")) \
    .map(lambda row: (row[2], 1)) \
    .groupByKey() \
    .mapValues(sum) \
    .sortByKey() \
    .collect()

print(class_travelers_grouped)

[('1', 216), ('2', 184), ('3', 491)]


In [ ]:
# Repartition the Titanic RDD to 15 partitions, then reduce it to 1

partitioned_rdd = titanic_rdd.repartition(15)
single_partition_rdd = partitioned_rdd.coalesce(1)

print(f"Final number of partitions: {single_partition_rdd.getNumPartitions()}")

Final number of partitions: 1


## PySpark DataFrame API

Using the PySpark DataFrame API for structured data processing, including selection, filtering, aggregation, grouping, and other DataFrame operations.

In [ ]:
# Read the Titanic dataset as a DataFrame

titanic_df = spark.read.format("parquet").load("titanic_parquet")

## Selecting DataFrame Columns

Using `select()` to retrieve and rename columns from a PySpark DataFrame.

In [ ]:
# Select survival status and passenger name

selected = titanic_df.select("Survived", "PassengerName")
selected.show()

+--------+--------------------+
|Survived|       PassengerName|
+--------+--------------------+
|       0|Braund, Mr. Owen ...|
|       1|Cumings, Mrs. Joh...|
|       1|Heikkinen, Miss. ...|
|       1|Futrelle, Mrs. Ja...|
|       0|Allen, Mr. Willia...|
|       0|    Moran, Mr. James|
|       0|McCarthy, Mr. Tim...|
|       0|Palsson, Master. ...|
|       1|Johnson, Mrs. Osc...|
|       1|Nasser, Mrs. Nich...|
|       1|Sandstrom, Miss. ...|
|       1|Bonnell, Miss. El...|
|       0|Saundercock, Mr. ...|
|       0|Andersson, Mr. An...|
|       0|Vestrom, Miss. Hu...|
|       1|Hewlett, Mrs. (Ma...|
|       0|Rice, Master. Eugene|
|       1|Williams, Mr. Cha...|
|       0|Vander Planke, Mr...|
|       1|Masselmani, Mrs. ...|
+--------+--------------------+
only showing top 20 rows


In [ ]:
# Select surviving passengers whose names contain "Mrs."

selected = titanic_df \
    .select("Survived", "PassengerName") \
    .filter(titanic_df.Survived == "1") \
    .filter(titanic_df.PassengerName.contains("Mrs."))

selected.show()

+--------+--------------------+
|Survived|       PassengerName|
+--------+--------------------+
|       1|Cumings, Mrs. Joh...|
|       1|Futrelle, Mrs. Ja...|
|       1|Johnson, Mrs. Osc...|
|       1|Nasser, Mrs. Nich...|
|       1|Hewlett, Mrs. (Ma...|
|       1|Masselmani, Mrs. ...|
|       1|Asplund, Mrs. Car...|
|       1|Spencer, Mrs. Wil...|
|       1|Harper, Mrs. Henr...|
|       1|Faunthorpe, Mrs. ...|
|       1|Nye, Mrs. (Elizab...|
|       1|Backstrom, Mrs. K...|
|       1|Doling, Mrs. John...|
|       1|Weisz, Mrs. Leopo...|
|       1|Hakkarainen, Mrs....|
|       1|Pears, Mrs. Thoma...|
|       1|Watt, Mrs. James ...|
|       1|Chibnall, Mrs. (E...|
|       1|O'Brien, Mrs. Tho...|
|       1| Pinsky, Mrs. (Rosa)|
+--------+--------------------+
only showing top 20 rows


In [ ]:
# Select and rename columns using alias()

selected = titanic_df.select(
    titanic_df["Survived"].alias("is_alive"),
    titanic_df["PassengerName"].alias("full_name")
)

selected.show()

+--------+--------------------+
|is_alive|           full_name|
+--------+--------------------+
|       0|Braund, Mr. Owen ...|
|       1|Cumings, Mrs. Joh...|
|       1|Heikkinen, Miss. ...|
|       1|Futrelle, Mrs. Ja...|
|       0|Allen, Mr. Willia...|
|       0|    Moran, Mr. James|
|       0|McCarthy, Mr. Tim...|
|       0|Palsson, Master. ...|
|       1|Johnson, Mrs. Osc...|
|       1|Nasser, Mrs. Nich...|
|       1|Sandstrom, Miss. ...|
|       1|Bonnell, Miss. El...|
|       0|Saundercock, Mr. ...|
|       0|Andersson, Mr. An...|
|       0|Vestrom, Miss. Hu...|
|       1|Hewlett, Mrs. (Ma...|
|       0|Rice, Master. Eugene|
|       1|Williams, Mr. Cha...|
|       0|Vander Planke, Mr...|
|       1|Masselmani, Mrs. ...|
+--------+--------------------+
only showing top 20 rows


In [ ]:
from pyspark.sql.functions import col

In [ ]:
# Select passenger names and ages using col()

titanic_df.select(
    col("PassengerName"),
    col("Age")
).show()

+--------------------+---+
|       PassengerName|Age|
+--------------------+---+
|Braund, Mr. Owen ...| 22|
|Cumings, Mrs. Joh...| 38|
|Heikkinen, Miss. ...| 26|
|Futrelle, Mrs. Ja...| 35|
|Allen, Mr. Willia...| 35|
|    Moran, Mr. James|   |
|McCarthy, Mr. Tim...| 54|
|Palsson, Master. ...|  2|
|Johnson, Mrs. Osc...| 27|
|Nasser, Mrs. Nich...| 14|
|Sandstrom, Miss. ...|  4|
|Bonnell, Miss. El...| 58|
|Saundercock, Mr. ...| 20|
|Andersson, Mr. An...| 39|
|Vestrom, Miss. Hu...| 14|
|Hewlett, Mrs. (Ma...| 55|
|Rice, Master. Eugene|  2|
|Williams, Mr. Cha...|   |
|Vander Planke, Mr...| 31|
|Masselmani, Mrs. ...|   |
+--------------------+---+
only showing top 20 rows


In [ ]:
# Select passenger names, ages, and embarkation ports

titanic_df.select(
    col("PassengerName"),
    col("Age"),
    col("Embarked")
).show()

+--------------------+---+--------+
|       PassengerName|Age|Embarked|
+--------------------+---+--------+
|Braund, Mr. Owen ...| 22|       S|
|Cumings, Mrs. Joh...| 38|       C|
|Heikkinen, Miss. ...| 26|       S|
|Futrelle, Mrs. Ja...| 35|       S|
|Allen, Mr. Willia...| 35|       S|
|    Moran, Mr. James|   |       Q|
|McCarthy, Mr. Tim...| 54|       S|
|Palsson, Master. ...|  2|       S|
|Johnson, Mrs. Osc...| 27|       S|
|Nasser, Mrs. Nich...| 14|       C|
|Sandstrom, Miss. ...|  4|       S|
|Bonnell, Miss. El...| 58|       S|
|Saundercock, Mr. ...| 20|       S|
|Andersson, Mr. An...| 39|       S|
|Vestrom, Miss. Hu...| 14|       S|
|Hewlett, Mrs. (Ma...| 55|       S|
|Rice, Master. Eugene|  2|       Q|
|Williams, Mr. Cha...|   |       S|
|Vander Planke, Mr...| 31|       S|
|Masselmani, Mrs. ...|   |       C|
+--------------------+---+--------+
only showing top 20 rows


In [ ]:
# Select passenger names, ages, and survival status

titanic_df.select(
    col("PassengerName"),
    col("Age"),
    col("Survived")
).show()

+--------------------+---+--------+
|       PassengerName|Age|Survived|
+--------------------+---+--------+
|Braund, Mr. Owen ...| 22|       0|
|Cumings, Mrs. Joh...| 38|       1|
|Heikkinen, Miss. ...| 26|       1|
|Futrelle, Mrs. Ja...| 35|       1|
|Allen, Mr. Willia...| 35|       0|
|    Moran, Mr. James|   |       0|
|McCarthy, Mr. Tim...| 54|       0|
|Palsson, Master. ...|  2|       0|
|Johnson, Mrs. Osc...| 27|       1|
|Nasser, Mrs. Nich...| 14|       1|
|Sandstrom, Miss. ...|  4|       1|
|Bonnell, Miss. El...| 58|       1|
|Saundercock, Mr. ...| 20|       0|
|Andersson, Mr. An...| 39|       0|
|Vestrom, Miss. Hu...| 14|       0|
|Hewlett, Mrs. (Ma...| 55|       1|
|Rice, Master. Eugene|  2|       0|
|Williams, Mr. Cha...|   |       1|
|Vander Planke, Mr...| 31|       0|
|Masselmani, Mrs. ...|   |       1|
+--------------------+---+--------+
only showing top 20 rows


## Filtering and Ordering DataFrames

Using `filter()`, `select()`, and `orderBy()` to query and sort Titanic passenger data.

In [ ]:
# Find passengers with more than 5 siblings or spouses on board

titanic_df.select(
    "PassengerName",
    "PassengerId",
    "SiblingsAndSpouses"
).filter(
    col("SiblingsAndSpouses").cast("int") > 5
).show()

+--------------------+-----------+------------------+
|       PassengerName|PassengerId|SiblingsAndSpouses|
+--------------------+-----------+------------------+
|Sage, Master. Tho...|        160|                 8|
|Sage, Miss. Const...|        181|                 8|
| Sage, Mr. Frederick|        202|                 8|
|Sage, Mr. George ...|        325|                 8|
|Sage, Miss. Stell...|        793|                 8|
|Sage, Mr. Douglas...|        847|                 8|
|Sage, Miss. Dorot...|        864|                 8|
+--------------------+-----------+------------------+



In [ ]:
# Find the passenger under 50 who paid the highest fare

from pyspark.sql.functions import desc

titanic_df \
    .filter(col("Age") != "") \
    .filter(col("Fare") != "") \
    .select(
        "PassengerName",
        col("Age").cast("double").alias("Age"),
        col("Fare").cast("double").alias("Fare")
    ) \
    .filter(col("Age") < 50) \
    .orderBy(desc("Fare")) \
    .limit(1) \
    .show()

+----------------+----+--------+
|   PassengerName| Age|    Fare|
+----------------+----+--------+
|Ward, Miss. Anna|35.0|512.3292|
+----------------+----+--------+



## Creating and Modifying DataFrame Columns

Using `withColumn()` to create new columns or transform existing columns in a PySpark DataFrame.

In [ ]:
# Add an age category column using withColumn() and when()

from pyspark.sql.functions import when

df_with_age_category = titanic_df.withColumn(
    "AgeCategory",
    when(col("Age") == "", "unknown")
    .when(col("Age").cast("double") < 18, "child")
    .otherwise("adult")
)

df_with_age_category.select("PassengerName", "Age", "AgeCategory").show()

+--------------------+---+-----------+
|       PassengerName|Age|AgeCategory|
+--------------------+---+-----------+
|Braund, Mr. Owen ...| 22|      adult|
|Cumings, Mrs. Joh...| 38|      adult|
|Heikkinen, Miss. ...| 26|      adult|
|Futrelle, Mrs. Ja...| 35|      adult|
|Allen, Mr. Willia...| 35|      adult|
|    Moran, Mr. James|   |    unknown|
|McCarthy, Mr. Tim...| 54|      adult|
|Palsson, Master. ...|  2|      child|
|Johnson, Mrs. Osc...| 27|      adult|
|Nasser, Mrs. Nich...| 14|      child|
|Sandstrom, Miss. ...|  4|      child|
|Bonnell, Miss. El...| 58|      adult|
|Saundercock, Mr. ...| 20|      adult|
|Andersson, Mr. An...| 39|      adult|
|Vestrom, Miss. Hu...| 14|      child|
|Hewlett, Mrs. (Ma...| 55|      adult|
|Rice, Master. Eugene|  2|      child|
|Williams, Mr. Cha...|   |    unknown|
|Vander Planke, Mr...| 31|      adult|
|Masselmani, Mrs. ...|   |    unknown|
+--------------------+---+-----------+
only showing top 20 rows


In [ ]:
# Multiply known passenger ages by -1

df_with_minus_age = titanic_df.withColumn(
    "Age",
    col("Age").try_cast("double") * -1
)

df_with_minus_age.select("PassengerName", "Age").show()

+--------------------+-----+
|       PassengerName|  Age|
+--------------------+-----+
|Braund, Mr. Owen ...|-22.0|
|Cumings, Mrs. Joh...|-38.0|
|Heikkinen, Miss. ...|-26.0|
|Futrelle, Mrs. Ja...|-35.0|
|Allen, Mr. Willia...|-35.0|
|    Moran, Mr. James| NULL|
|McCarthy, Mr. Tim...|-54.0|
|Palsson, Master. ...| -2.0|
|Johnson, Mrs. Osc...|-27.0|
|Nasser, Mrs. Nich...|-14.0|
|Sandstrom, Miss. ...| -4.0|
|Bonnell, Miss. El...|-58.0|
|Saundercock, Mr. ...|-20.0|
|Andersson, Mr. An...|-39.0|
|Vestrom, Miss. Hu...|-14.0|
|Hewlett, Mrs. (Ma...|-55.0|
|Rice, Master. Eugene| -2.0|
|Williams, Mr. Cha...| NULL|
|Vander Planke, Mr...|-31.0|
|Masselmani, Mrs. ...| NULL|
+--------------------+-----+
only showing top 20 rows


## Renaming DataFrame Columns

Using `withColumnRenamed()` to rename individual or multiple columns in a PySpark DataFrame.

In [ ]:
# Rename the Embarked column

renamed_embarked_df = titanic_df.withColumnRenamed(
    "Embarked",
    "Port_of_Embarkation"
)

renamed_embarked_df.printSchema()

root
 |-- PassengerId: string (nullable = true)
 |-- Survived: string (nullable = true)
 |-- PassengerClass: string (nullable = true)
 |-- PassengerName: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- SiblingsAndSpouses: string (nullable = true)
 |-- ParentsAndChildren: string (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: string (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Port_of_Embarkation: string (nullable = true)



In [ ]:
# Rename the SiblingsAndSpouses column

renamed_siblings_df = titanic_df.withColumnRenamed(
    "SiblingsAndSpouses",
    "Siblings_or_Spouses_on_Board"
)

renamed_siblings_df.printSchema()

root
 |-- PassengerId: string (nullable = true)
 |-- Survived: string (nullable = true)
 |-- PassengerClass: string (nullable = true)
 |-- PassengerName: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Siblings_or_Spouses_on_Board: string (nullable = true)
 |-- ParentsAndChildren: string (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: string (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [ ]:
# Rename the ParentsAndChildren column

renamed_parents_df = titanic_df.withColumnRenamed(
    "ParentsAndChildren",
    "Parents_or_Children_on_Board"
)

renamed_parents_df.printSchema()

root
 |-- PassengerId: string (nullable = true)
 |-- Survived: string (nullable = true)
 |-- PassengerClass: string (nullable = true)
 |-- PassengerName: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- SiblingsAndSpouses: string (nullable = true)
 |-- Parents_or_Children_on_Board: string (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: string (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [ ]:
# Rename multiple columns

renamed_df = titanic_df \
    .withColumnRenamed("Age", "NewAge") \
    .withColumnRenamed("PassengerName", "FullName")

renamed_df.printSchema()

root
 |-- PassengerId: string (nullable = true)
 |-- Survived: string (nullable = true)
 |-- PassengerClass: string (nullable = true)
 |-- FullName: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- NewAge: string (nullable = true)
 |-- SiblingsAndSpouses: string (nullable = true)
 |-- ParentsAndChildren: string (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: string (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [ ]:
# Rename all columns with a "Col_" prefix

new_column_names = [f"Col_{i}" for i in range(len(titanic_df.columns))]

renamed_all_df = titanic_df.toDF(*new_column_names)

renamed_all_df.printSchema()

root
 |-- Col_0: string (nullable = true)
 |-- Col_1: string (nullable = true)
 |-- Col_2: string (nullable = true)
 |-- Col_3: string (nullable = true)
 |-- Col_4: string (nullable = true)
 |-- Col_5: string (nullable = true)
 |-- Col_6: string (nullable = true)
 |-- Col_7: string (nullable = true)
 |-- Col_8: string (nullable = true)
 |-- Col_9: string (nullable = true)
 |-- Col_10: string (nullable = true)
 |-- Col_11: string (nullable = true)



## Grouping and Aggregation

Using `groupBy()` and `agg()` to group Titanic passenger data and calculate aggregate statistics.

In [ ]:
# Calculate the maximum age by sex and passenger class

from pyspark.sql.functions import max

max_age_df = titanic_df \
    .filter(col("Age") != "") \
    .groupBy("Sex", "PassengerClass") \
    .agg(max(col("Age").cast("double")).alias("MaxAge")) \
    .orderBy("Sex", "PassengerClass")

max_age_df.show()


+------+--------------+------+
|   Sex|PassengerClass|MaxAge|
+------+--------------+------+
|female|             1|  63.0|
|female|             2|  57.0|
|female|             3|  63.0|
|  male|             1|  80.0|
|  male|             2|  70.0|
|  male|             3|  74.0|
+------+--------------+------+



In [ ]:
# Calculate average age and maximum fare by sex

from pyspark.sql.functions import avg, max

titanic_df \
    .filter((col("Age") != "") & (col("Fare") != "")) \
    .groupBy("Sex") \
    .agg(
        avg(col("Age").cast("double")).alias("AverageAge"),
        max(col("Fare").cast("double")).alias("MaxFare")
    ) \
    .show()

+------+------------------+--------+
|   Sex|        AverageAge| MaxFare|
+------+------------------+--------+
|female|27.915708812260537|512.3292|
|  male| 30.72664459161148|512.3292|
+------+------------------+--------+



In [ ]:
# Calculate the average fare by passenger class

titanic_df \
    .filter(col("Fare") != "") \
    .groupBy("PassengerClass") \
    .agg(
        avg(col("Fare").cast("double")).alias("AverageFare")
    ) \
    .orderBy("PassengerClass") \
    .show()

+--------------+------------------+
|PassengerClass|       AverageFare|
+--------------+------------------+
|             1| 84.15468749999992|
|             2| 20.66218315217391|
|             3|13.675550101832997|
+--------------+------------------+



## Joining DataFrames

Using `join()` to combine PySpark DataFrames based on a common key.

In [ ]:
# Create an age DataFrame and join it with the Titanic DataFrame by PassengerId

age_df = titanic_df.select(
    "PassengerId",
    col("Age").alias("JoinedAge")
)

joined_df = titanic_df.join(
    age_df,
    on="PassengerId",
    how="inner"
)

joined_df.select(
    "PassengerId",
    "PassengerName",
    "Age",
    "JoinedAge"
).show()


+-----------+--------------------+---+---------+
|PassengerId|       PassengerName|Age|JoinedAge|
+-----------+--------------------+---+---------+
|          1|Braund, Mr. Owen ...| 22|       22|
|          2|Cumings, Mrs. Joh...| 38|       38|
|          3|Heikkinen, Miss. ...| 26|       26|
|          4|Futrelle, Mrs. Ja...| 35|       35|
|          5|Allen, Mr. Willia...| 35|       35|
|          6|    Moran, Mr. James|   |         |
|          7|McCarthy, Mr. Tim...| 54|       54|
|          8|Palsson, Master. ...|  2|        2|
|          9|Johnson, Mrs. Osc...| 27|       27|
|         10|Nasser, Mrs. Nich...| 14|       14|
|         11|Sandstrom, Miss. ...|  4|        4|
|         12|Bonnell, Miss. El...| 58|       58|
|         13|Saundercock, Mr. ...| 20|       20|
|         14|Andersson, Mr. An...| 39|       39|
|         15|Vestrom, Miss. Hu...| 14|       14|
|         16|Hewlett, Mrs. (Ma...| 55|       55|
|         17|Rice, Master. Eugene|  2|        2|
|         18|William

In [ ]:
# Calculate the minimum age by sex and passenger class

from pyspark.sql.functions import min

min_age_df = titanic_df \
    .filter(col("Age") != "") \
    .groupBy("Sex", "PassengerClass") \
    .agg(min(col("Age").cast("double")).alias("MinAge")) \
    .orderBy("Sex", "PassengerClass")

min_age_df.show()

+------+--------------+------+
|   Sex|PassengerClass|MinAge|
+------+--------------+------+
|female|             1|   2.0|
|female|             2|   2.0|
|female|             3|  0.75|
|  male|             1|  0.92|
|  male|             2|  0.67|
|  male|             3|  0.42|
+------+--------------+------+

